# Lezione 5 — CNN per il riconoscimento di immagini

**Obiettivo:** costruire una CNN minimale su MNIST e collegare preprocessing immagini, rete convoluzionale, training e valutazione.

> Se stai lavorando in Colab o su una macchina nuova, potrebbe servire:
> `pip install torch torchvision`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
import pandas as pd


## 1) Dataset MNIST

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds_full = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# subset piccolo per velocizzare il laboratorio
train_ds = Subset(train_ds_full, range(10000))
test_ds = Subset(test_ds_full, range(2000))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(8, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(images[i][0], cmap="gray")
    plt.title(f"label={labels[i].item()}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2) CNN minimale

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
model

## 3) Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += xb.size(0)

    return running_loss / total, correct / total

In [ ]:
for epoch in range(2):
    loss, acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Epoch {epoch+1} - loss: {loss:.4f} - acc: {acc:.4f}")

## 4) Valutazione

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            pred = logits.argmax(dim=1).cpu().numpy()

            all_preds.extend(pred)
            all_true.extend(yb.numpy())

    return np.array(all_true), np.array(all_preds)

y_true, y_pred = evaluate(model, test_loader, device)
test_acc = (y_true == y_pred).mean()
test_acc

In [ ]:
cm = confusion_matrix(y_true, y_pred)
pd.DataFrame(cm)

## 5) Esempi sbagliati

In [ ]:
wrong_idx = np.where(y_true != y_pred)[0][:8]

# recupero immagini dal subset test_ds
plt.figure(figsize=(8, 4))
for i, idx in enumerate(wrong_idx):
    img, label = test_ds[idx]
    plt.subplot(2, 4, i + 1)
    plt.imshow(img[0], cmap="gray")
    plt.title(f"t={label}, p={y_pred[idx]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6) Comunicazione dei risultati

Scrivi una conclusione:
1. Che preprocessing abbiamo applicato alle immagini?
2. Qual è il ruolo di convoluzione e pooling?
3. Dove sbaglia la CNN?
4. Quale miglioramento proveresti per primo?